In [1]:
#type: ignore
import ee
import geemap
import eemont

In [2]:
ee.Authenticate()
ee.Initialize(project="ee-dayronlalay8")

In [10]:

unidades = ee.FeatureCollection("FAO/GAUL/2015/level2")

Map = geemap.Map(basemap="CartoDB.DarkMatter", center=[6.648424, 52.869798], zoom=6)

netherlands = unidades.filter(ee.Filter.eq('ADM0_NAME', 'Netherlands'))

# Obtener lista de provincias (ADM1_NAME)
provincias = netherlands.aggregate_array('ADM1_NAME').distinct()

print('Provincias:', provincias.getInfo()) 

# Filtrar la provincia de Drenthe (ADM1_NAME)
aoi = unidades.filter(ee.Filter.eq('ADM1_NAME', 'Drenthe'))

Map.addLayer(aoi, {'color': 'blue'}, 'Drenthe (Provincia)')
Map.centerObject(aoi, 9)

Map

Provincias: ['Drenthe', 'Flevoland', 'Friesland', 'Gelderland', 'Groningen', 'Limburg', 'Noord-brabant', 'Noord-holland', 'Overijssel', 'Utrecht', 'Zeeland', 'Zuid-holland']


Map(center=[52.86164841483282, 6.620470303838324], controls=(WidgetControl(options=['position', 'transparent_b…

In [14]:
dataset = ee.Image('AHN/AHN4/AHN4_20200101')
dataset

In [28]:
anh = (ee.ImageCollection("AHN/AHN4")
       .filterBounds(aoi)
       .mosaic())

elevation = anh.select('dtm').clip(aoi)

image = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterBounds(aoi)
           .filterDate("2025-08-01", "2025-09-01")
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
           .preprocess()
           .spectralIndices(["NDVI"])
           .mean()
           .clip(aoi))

In [34]:
Map = geemap.Map(basemap="CartoDB.DarkMatter", center=[6.648424, 52.869798], zoom=6)


stats = elevation.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=aoi,
    scale=20,
    bestEffort=True
)

min_val = stats.get('dtm_min')  # ajusta según nombre de banda
max_val = stats.get('dtm_max')

ndvi = ee.ImageCollection([image.select("NDVI")])

# Visualización con rango calculado

elevation_visual = {
    'min': min_val, 
    'max': max_val, 
    'palette': ['green', 'yellow', 'brown', 'white']
}

visualization = {
    'min': 0.0,
    'max': 0.3,
    'bands': ["B4","B3","B2"],
}

palette_ndvi = [
    "#f4f4f7",  # sin dato
    "#d73027",  # sin vegetación
    "#f46d43",  # suelo desnudo
    "#fee08b",  # vegetación débil
    "#a6d96a",  # vegetación media
    "#1a9850",  # vegetación densa
]

params_ndvi = {"min": -1, "max": 0.9, "palette": palette_ndvi}



Map.addLayer(aoi, {'color': 'blue'}, 'Drenthe (Provincia)')
Map.addLayer(image, visualization, 'RGB Sentinel-2')
Map.addLayer(elevation, elevation_visual, 'DEM')
Map.addLayer(ndvi, params_ndvi, 'NDVI')
Map.centerObject(aoi, 9)
Map

Map(center=[52.861648414832764, 6.6204703038383315], controls=(WidgetControl(options=['position', 'transparent…